# Tutorial 2: Basis Set Study

**Comparing accuracy vs. computational cost**

## Learning Objectives
- Understand what a basis set is and why it matters
- Compare STO-3G, 6-31G, 6-31G* and cc-pVDZ on the same molecule
- Interpret convergence patterns in total energy
- Balance accuracy with computational cost

## Time: 30–35 minutes (the calculations take a few seconds)

---

### Background

A **basis set** is the mathematical language we use to describe molecular orbitals:
each orbital is built as a combination of these functions. More (and more
flexible) functions describe the electrons better, but the calculation gets
more expensive.

| Basis set | Size | Speed | Accuracy |
|-----------|------|-------|----------|
| STO-3G    | Minimal | Very fast | Low |
| 6-31G     | Split-valence | Fast | Moderate |
| 6-31G*    | Split-valence + polarization (d on O) | Fast | Better |
| cc-pVDZ   | Correlation-consistent double-zeta (polarized) | Slower | Higher |

We will run **RHF** (Restricted Hartree-Fock) on water (H₂O) with each basis set,
at the same geometry, and compare the total energies.

**The variational principle:** for Hartree-Fock, a more complete basis can only
lower (or keep) the energy. So the *lowest* energy here is the best description
of the RHF limit — not just a different number.

In [ ]:
# QuantUI — in-session calculation imports
import io
import time

from quantui import Molecule, parse_xyz_input

try:
    from quantui import run_in_session

    PYSCF_AVAILABLE = True
except ImportError:
    PYSCF_AVAILABLE = False
    print("PySCF not available — calculations will not run.")
    print("PySCF requires Linux, macOS, or WSL.")

In [ ]:
# Define the water molecule (same geometry as Tutorial 1)
water_xyz = """O  0.000  0.000  0.000
H  0.757  0.587  0.000
H -0.757  0.587  0.000"""

atoms, coords = parse_xyz_input(water_xyz)
molecule = Molecule(atoms, coords, charge=0, multiplicity=1)
print(f"Molecule: {molecule.get_formula()} ready")

### Run the four calculations

They run one after another in this Python session. `warm_start=False` makes
each one start from scratch, so the timings and iteration counts are fair.

In [ ]:
basis_sets = ["STO-3G", "6-31G", "6-31G*", "cc-pVDZ"]
basis_results = {}  # basis name -> SessionResult
timings = {}  # basis name -> seconds
n_functions = {}  # basis name -> number of basis functions

if not PYSCF_AVAILABLE:
    print("Skipping calculations — PySCF not available.")
else:
    from pyscf import gto

    for basis in basis_sets:
        n_functions[basis] = gto.M(
            atom=molecule.to_pyscf_format(), basis=basis, verbose=0
        ).nao
        start = time.perf_counter()
        result = run_in_session(
            molecule,
            method="RHF",
            basis=basis,
            progress_stream=io.StringIO(),
            warm_start=False,
        )
        timings[basis] = time.perf_counter() - start
        basis_results[basis] = result
        print(
            f"RHF/{basis:8s} {result.energy_hartree:.6f} Ha  "
            f"({n_functions[basis]} basis functions, {timings[basis]:.2f} s)"
        )
    print("\nAll calculations complete.")

In [ ]:
# Compare energies (Hartree and kcal/mol relative to cc-pVDZ)
HARTREE_TO_KCAL = 627.5095

if basis_results:
    reference = basis_results["cc-pVDZ"].energy_hartree
    print(f"{'Basis set':10s} {'functions':>9s} {'Energy (Ha)':>13s} {'vs cc-pVDZ (kcal/mol)':>23s}")
    print("-" * 60)
    for basis in basis_sets:
        energy = basis_results[basis].energy_hartree
        diff_kcal = (energy - reference) * HARTREE_TO_KCAL
        print(f"{basis:10s} {n_functions[basis]:>9d} {energy:>13.6f} {diff_kcal:>+23.1f}")
else:
    print("No results to compare. Run the cell above first.")

## Complete!

**What you should see** (energies above the cc-pVDZ result, at this geometry):
- **STO-3G** (7 basis functions): about **+667 kcal/mol** — a minimal basis
  cannot describe the electrons well at all
- **6-31G** (13 functions): about **+27 kcal/mol** — a huge improvement
- **6-31G\*** (18 functions): about **+11 kcal/mol** — adding d functions on
  oxygen (polarization) helps noticeably
- **cc-pVDZ** (24 functions): the lowest energy here, so the best of the four

The energy keeps dropping as the basis grows, exactly as the variational
principle says. Even cc-pVDZ is not the complete-basis limit: larger sets
(cc-pVTZ, …) lower it further. On a molecule this small every calculation takes
well under a second; the cost difference becomes real for bigger molecules,
because the work grows roughly as the fourth power of the number of basis
functions.

**Important:** absolute energies are only comparable at the *same* basis set
and method. Chemistry comes from *differences* (reaction energies, barriers)
computed consistently, and those usually converge faster with basis size than
the total energies do.

**Questions to explore:**
1. Which basis set gives the lowest (most negative) energy? Why?
2. How does the number of basis functions grow from STO-3G to cc-pVDZ?
3. For a quick check, is STO-3G acceptable? For publication quality?

**Next:** [Tutorial 3: Multiplicity and Radicals](03_multiplicity_radicals.ipynb)